# Data Preprocessing — Qué Cocinar IA

Este notebook carga `data/recipes.csv`, limpia los datos y los indexa en ChromaDB.

In [4]:
import ast
import re
from pathlib import Path

import pandas as pd
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

PROJECT_ROOT = Path("..").resolve()
CSV_PATH = PROJECT_ROOT / "data" / "recipes.csv"
CHROMA_DIR = PROJECT_ROOT / "chroma_db"
COLLECTION_NAME = "recipes"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
BATCH_SIZE = 500

In [5]:
# Load local recipes dataset
if not CSV_PATH.exists():
    raise FileNotFoundError(f"No se encontró el archivo: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
print(f"Archivo: {CSV_PATH}")
print(f"Filas: {len(df):,} | Columnas: {list(df.columns)}")
df.head(2)

Archivo: /Users/isabellabresciani_1/Documents/que-cocinar-IA/que-cocinar-IA/data/recipes.csv
Filas: 1,090 | Columnas: ['Unnamed: 0', 'recipe_name', 'prep_time', 'cook_time', 'total_time', 'servings', 'yield', 'ingredients', 'directions', 'rating', 'url', 'cuisine_path', 'nutrition', 'timing', 'img_src']


,Unnamed: 0,recipe_name,prep_time,cook_time,total_time,servings,yield,ingredients,directions,rating,url,cuisine_path,nutrition,timing,img_src
0,0,Apple-Cranberry Crostada,NaN,NaN,NaN,8,6 to 8 - servings,"3 tablespoons butter, 2 pounds Granny Smith ap...",Heat butter in a large skillet over medium-hig...,4.4,https://www.allrecipes.com/recipe/76931/apple-...,/Desserts/Fruit Desserts/Apple Dessert Recipes/,"Total Fat 18g 23%, Saturated Fat 7g 34%, Chole...","Servings: 8, Yield: 6 to 8 - servings",https://www.allrecipes.com/thmb/Tf1wH73bfH6Oql...
1,1,Apple Pie by Grandma Ople,30 mins,1 hrs,1 hrs 30 mins,8,1 9-inch pie,"8 small Granny Smith apples, or as needed, ½ c...","Peel and core apples, then thinly slice. Set a...",4.8,https://www.allrecipes.com/recipe/12682/apple-...,/Desserts/Pies/Apple Pie Recipes/,"Total Fat 19g 24%, Saturated Fat 9g 46%, Chole...","Prep Time: 30 mins, Cook Time: 1 hrs, Total Ti...",https://www.allrecipes.com/thmb/1I95oiTGz6aEpu...


In [6]:
def time_to_minutes(value) -> int | None:
    """Parse time strings like '25 mins', '1 hr 10 mins' into integer minutes."""
    if pd.isna(value) or value == "":
        return None
    text = str(value).lower().strip()
    hours = 0
    minutes = 0
    hr_match = re.search(r"(\d+)\s*h(?:r|our|ora)?", text)
    min_match = re.search(r"(\d+)\s*m(?:in|inute|inuto)?", text)
    if hr_match:
        hours = int(hr_match.group(1))
    if min_match:
        minutes = int(min_match.group(1))
    if not hr_match and not min_match:
        digits = re.findall(r"\d+", text)
        if digits:
            minutes = int(digits[0])
        else:
            return None
    return hours * 60 + minutes


def parse_servings(value) -> int | None:
    if pd.isna(value) or value == "":
        return None
    digits = re.findall(r"\d+", str(value))
    return int(digits[0]) if digits else None


def parse_nutrition(value) -> dict:
    """Safely parse nutrition string-dict and flatten to numeric fields."""
    result = {"calories": None, "protein_g": None, "carbs_g": None, "fat_g": None}
    if pd.isna(value) or value == "":
        return result
    try:
        data = ast.literal_eval(str(value)) if isinstance(value, str) else value
    except (ValueError, SyntaxError):
        return result
    if not isinstance(data, dict):
        return result

    key_map = {
        "calories": "calories",
        "proteinContent": "protein_g",
        "carbohydrateContent": "carbs_g",
        "fatContent": "fat_g",
    }
    for src, dst in key_map.items():
        raw = data.get(src)
        if raw is None:
            continue
        nums = re.findall(r"[\d.]+", str(raw))
        if nums:
            result[dst] = float(nums[0])
    return result


# Drop rows missing critical columns
critical = ["recipe_name", "ingredients", "directions"]
for col in critical:
    if col not in df.columns:
        raise KeyError(f"Columna requerida ausente: {col}")

df = df.dropna(subset=critical).copy()
df = df[df["recipe_name"].str.strip() != ""]
df = df[df["ingredients"].str.strip() != ""]
df = df[df["directions"].str.strip() != ""]

# Stable id to trace each recipe back to data/recipes.csv
if "Unnamed: 0" in df.columns:
    df["csv_row_id"] = df["Unnamed: 0"].astype(int)
else:
    df["csv_row_id"] = df.index.astype(int)

print(f"Filas tras limpieza: {len(df):,}")

Filas tras limpieza: 1,090


In [7]:
def row_to_document(row) -> Document | None:
    """Convert a DataFrame row into a LangChain Document with scalar metadata."""
    name = str(row["recipe_name"]).strip()
    ingredients = str(row["ingredients"]).strip()
    directions = str(row["directions"]).strip()

    page_content = (
        f"{name}\n\n"
        f"Ingredientes:\n{ingredients}\n\n"
        f"Preparación:\n{directions}"
    )

    nutrition = parse_nutrition(row.get("nutrition"))

    metadata = {
        "csv_row_id": int(row["csv_row_id"]),
        "recipe_name": name,
        "prep_time_min": time_to_minutes(row.get("prep_time")),
        "cook_time_min": time_to_minutes(row.get("cook_time")),
        "total_time_min": time_to_minutes(row.get("total_time")),
        "servings": parse_servings(row.get("servings")),
        "rating": float(row["rating"]) if pd.notna(row.get("rating")) else None,
        "cuisine_path": str(row.get("cuisine_path", "")) if pd.notna(row.get("cuisine_path")) else "",
        **nutrition,
    }

    # Chroma only accepts str, int, float, bool — drop None values
    metadata = {k: v for k, v in metadata.items() if v is not None}
    return Document(page_content=page_content, metadata=metadata)


documents = [doc for doc in (row_to_document(row) for _, row in df.iterrows()) if doc]
print(f"Documentos listos: {len(documents):,}")
print(documents[0].page_content[:300], "...\n")
print("Metadata ejemplo:", documents[0].metadata)

Documentos listos: 1,090
Apple-Cranberry Crostada

Ingredientes:
3 tablespoons butter, 2 pounds Granny Smith apples (or other firm, crisp apples), peeled, quartered, cored and sliced 1/4-inch thick, 1 pound Macintosh apples (or other soft-textured apples that fall apart when cooked), peeled, quartered, cored, and sliced 1/4 ...

Metadata ejemplo: {'recipe_name': 'Apple-Cranberry Crostada', 'servings': 8, 'rating': 4.4, 'cuisine_path': '/Desserts/Fruit Desserts/Apple Dessert Recipes/'}


In [11]:
# Embed and persist to ChromaDB (batched to limit memory)
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

CHROMA_DIR.mkdir(parents=True, exist_ok=True)

vectorstore = None
for i in range(0, len(documents), BATCH_SIZE):
    batch = documents[i : i + BATCH_SIZE]
    if vectorstore is None:
        vectorstore = Chroma.from_documents(
            documents=batch,
            embedding=embeddings,
            collection_name=COLLECTION_NAME,
            persist_directory=str(CHROMA_DIR),
        )
    else:
        vectorstore.add_documents(batch)
    print(f"Indexados {min(i + BATCH_SIZE, len(documents)):,} / {len(documents):,}")

print(f"\nChromaDB persistido en: {CHROMA_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexados 500 / 1,090
Indexados 1,000 / 1,090
Indexados 1,090 / 1,090

ChromaDB persistido en: /Users/isabellabresciani_1/Documents/que-cocinar-IA/que-cocinar-IA/chroma_db


In [12]:
# Smoke tests
results = vectorstore.similarity_search("chicken rice", k=2)
print("=== Búsqueda semántica: chicken rice ===")
for r in results:
    print(f"- {r.metadata.get('recipe_name')} ({r.metadata.get('total_time_min', '?')} min)")

quick = vectorstore.similarity_search(
    "quick easy meal",
    k=2,
    filter={"total_time_min": {"$lte": 20}},
)
print("\n=== Filtro metadata: total_time_min <= 20 ===")
for r in quick:
    print(f"- {r.metadata.get('recipe_name')} ({r.metadata.get('total_time_min')} min)")

=== Búsqueda semántica: chicken rice ===
- Mexican Chicken and Rice Soup (Sopa de Pollo y Arroz) (70 min)
- Mexican Chicken and Rice Soup (Sopa de Pollo y Arroz) (70 min)

=== Filtro metadata: total_time_min <= 20 ===
- Easy Mango Salad (20 min)
- Easy Mango Salad (20 min)
